In [ ]:
import json
import os
import pickle
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split

from app.nlp.classifier import (
    train_tfidf_classifier,
    train_embedding_classifier,
    predict_tactic,
    compare_classifiers,
)
from config import Config

sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
with open(Config.STIX_FEED_PATH) as f:
    bundles = json.load(f)

texts = []
labels = []
for bundle in bundles:
    texts.append(bundle['metadata']['raw_text'])
    labels.append(bundle['metadata']['tactic'])

print(f'Total samples: {len(texts)}')
print(f'Unique tactics: {len(set(labels))}')
print(f'Tactic counts: {dict(zip(*np.unique(labels, return_counts=True)))}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.3, random_state=42, stratify=labels
)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

In [ ]:
print('Training TF-IDF classifier...')
tfidf_model = train_tfidf_classifier(X_train, y_train)
print('TF-IDF model trained.')

In [ ]:
print('Training embedding classifier...')
embed_model = train_embedding_classifier(X_train, y_train)
print('Embedding model trained.')

In [ ]:
results = compare_classifiers(X_test, y_test, tfidf_model, embed_model)

## F1 Score per Tactic

In [ ]:
labels_sorted = sorted(set(labels))
tfidf_f1 = [results['tfidf']['report'].get(lbl, {}).get('f1-score', 0) for lbl in labels_sorted]
embed_f1 = [results['embedding']['report'].get(lbl, {}).get('f1-score', 0) for lbl in labels_sorted]

x = np.arange(len(labels_sorted))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - width/2, tfidf_f1, width, label='TF-IDF', color='#2196F3')
ax.bar(x + width/2, embed_f1, width, label='Sentence Embedding', color='#4CAF50')
ax.set_ylabel('F1 Score')
ax.set_title('F1 Score per Tactic')
ax.set_xticks(x)
ax.set_xticklabels(labels_sorted, rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## Confusion Matrix — TF-IDF

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(results['tfidf']['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
            xticklabels=tfidf_model.classes_, yticklabels=tfidf_model.classes_, ax=ax)
ax.set_title('Confusion Matrix — TF-IDF')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.tight_layout()
plt.show()

## Confusion Matrix — Sentence Embeddings

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(results['embedding']['confusion_matrix'], annot=True, fmt='d', cmap='Greens',
            xticklabels=embed_model['clf'].classes_, yticklabels=embed_model['clf'].classes_, ax=ax)
ax.set_title('Confusion Matrix — Sentence Embeddings')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.tight_layout()
plt.show()

In [ ]:
os.makedirs('../outputs', exist_ok=True)

with open('../outputs/tfidf_model.pkl', 'wb') as f:
    pickle.dump(tfidf_model, f)

with open('../outputs/embed_model.pkl', 'wb') as f:
    pickle.dump(embed_model, f)

print('Models saved to outputs/')

In [ ]:
sample_text = 'Detected outbound traffic to known C2 IP over port 443'
tactic_tfidf, conf_tfidf = predict_tactic(tfidf_model, sample_text, 'tfidf')
tactic_embed, conf_embed = predict_tactic(embed_model, sample_text, 'embedding')

print(f'TF-IDF     -> {tactic_tfidf:30s} confidence={conf_tfidf:.3f}')
print(f'Embedding  -> {tactic_embed:30s} confidence={conf_embed:.3f}')